# 12 Security Institutional Ties Development
## Verified membership reconstruction

### Purpose

This notebook replaces the wide treaty workbook as the candidate production source for Security Institutional ties. The original workbook remains a research register and comparison input. A normalized, source-backed membership register now controls annual membership spells and relationship depth.

The statistical object is **verified formal security cooperation between a country and each anchor in a given year**. The output is an anchor-specific non-negative score. It is not yet the combined Security pillar and is not yet combined with SIPRI arms-transfer orientation.

## Decisions applied in this revision

- Use institutional **depth**, not a count of arrangements, as the headline measure.
- Use an anchor-neutral scale: `3` alliance or mutual-defence commitment; `2` formal and durable security cooperation; `1` structured security participation; `0` observed no qualifying relationship.
- Use the maximum verified depth within each country-anchor-year to avoid double-counting overlapping institutions.
- Retain every supporting arrangement and relationship type for audit.
- Score full Shanghai Cooperation Organisation membership at depth `1`; omit observer and dialogue-partner statuses from the baseline.
- Preserve real US–China institutional asymmetry. Do not add generic strategic partnerships merely to balance the anchors.
- Treat the 2004 tsunami-response coordination as a Quad precursor, not formal Quad membership. Use the 2007–2008 and 2017–2025 institutional phases.
- Treat AUKUS as formal durable defence cooperation at depth `2`, beginning in 2021.
- Exclude the US–India Civil Nuclear Agreement from the baseline pending a separate strategic-technology decision.
- Keep disputed, special, and non-sovereign entities outside the baseline denominator pending an index-wide entity policy.

## Correction record

The first development run exposed material errors in the original workbook.

- **NATO:** later accessions had been backfilled to 2004 and Honduras was incorrectly marked as a NATO member. This revision uses official NATO accession years.
- **Rio Treaty:** several withdrawal periods were encoded in reverse. This revision uses verified membership spells and effective cessation years.
- **Quad:** 2004 humanitarian coordination had been treated as membership. This revision uses the formal 2007–2008 phase and the revived phase from 2017.
- **AUKUS:** the workbook began in 2023. This revision begins in 2021, when the partnership was announced.
- **US security-status coverage:** the original workbook omitted several current Major Non-NATO Allies and mixed MNNA status with separate mutual-defence commitments. This revision records the formal status and assigns depth `3` only where a separate mutual-defence treaty supports alliance depth.
- **China:** the generic treaty-ally category is replaced by the named China–DPRK treaty. Full SCO membership is retained at depth `1`, and the 2022 China–Solomon Islands Security Cooperation Framework is added at depth `2`.

Every workbook-versus-register disagreement is exported rather than silently overwritten.

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from openpyxl import load_workbook

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")


PROJECT_ROOT = find_project_root()
WORKBOOK_PATH = PROJECT_ROOT / "data" / "sources" / "relations" / "US-China Index Treaties.xlsx"
REGISTER_PATH = PROJECT_ROOT / "data" / "sources" / "relations" / "security_arrangement_membership_register.csv"
COUNTRY_MASTER_PATH = PROJECT_ROOT / "outputs" / "tables" / "relations_condition" / "relationship_condition_country_master.csv"
RELATIONSHIP_CROSSWALK_PATH = PROJECT_ROOT / "outputs" / "tables" / "relations_condition" / "relationship_country_crosswalk_audit.csv"
SOURCE_ALIAS_PATH = PROJECT_ROOT / "outputs" / "tables" / "relations_condition" / "country_crosswalk_audit.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables" / "security_institutional_ties"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required = [WORKBOOK_PATH, REGISTER_PATH, COUNTRY_MASTER_PATH, RELATIONSHIP_CROSSWALK_PATH, SOURCE_ALIAS_PATH]
missing = [str(p.relative_to(PROJECT_ROOT)) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")

for p in required:
    print(p.relative_to(PROJECT_ROOT))
print(f"Output directory: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")

data/sources/relations/US-China Index Treaties.xlsx
data/sources/relations/security_arrangement_membership_register.csv
outputs/tables/relations_condition/relationship_condition_country_master.csv
outputs/tables/relations_condition/relationship_country_crosswalk_audit.csv
outputs/tables/relations_condition/country_crosswalk_audit.csv
Output directory: outputs/tables/security_institutional_ties


## 1. Validate the verified membership register

One row represents one verified membership or agreement spell. Blank end years are not used in this version; continuing spells are explicitly bounded at 2025, the end of the current source window.

In [2]:
register = pd.read_csv(REGISTER_PATH)
required_register_columns = {
    "arrangement_id", "arrangement_name", "country_iso3", "anchor", "start_year", "end_year",
    "depth", "relationship_type", "source_url", "source_note", "review_status",
}
assert required_register_columns.issubset(register.columns)
assert register["anchor"].isin(["China", "United States"]).all()
assert register["depth"].isin([1, 2, 3]).all()
assert register["start_year"].le(register["end_year"]).all()
assert register["source_url"].notna().all()
assert register["review_status"].eq("verified").all()
assert not register.duplicated(["arrangement_id", "country_iso3", "anchor", "start_year", "end_year"]).any()

print(f"Membership spells: {len(register):,}")
print(f"Arrangements: {register['arrangement_id'].nunique():,}")
print(register.groupby(["anchor", "arrangement_id", "arrangement_name"], as_index=False).size().to_string(index=False))

Membership spells: 96
Arrangements: 10
       anchor         arrangement_id                                                   arrangement_name  size
        China      china_dprk_treaty China-DPRK Treaty of Friendship, Cooperation and Mutual Assistance     1
        China china_solomon_security               China-Solomon Islands Security Cooperation Framework     1
        China        sco_full_member                  Shanghai Cooperation Organisation full membership     9
United States                  aukus                                                              AUKUS     2
United States              five_eyes                                                          Five Eyes     4
United States                   mnna             Major Non-NATO Ally status and associated treaty depth    19
United States                   nato                                 North Atlantic Treaty Organization    31
United States                   quad                                    Quadrilat

## 2. Reconstruct and classify the country universe

The established project country master contains several historical entities incorrectly marked as current and several policy-dependent or special entities. The notebook exports a correction proposal and excludes them from this component's baseline denominator transparently.

In [3]:
country_master = pd.read_csv(COUNTRY_MASTER_PATH)

# Historical codes with no valid overlap with 2004-2025. The canonical
# country master currently marks several of these as non-historical, so the
# audit records a proposed upstream correction rather than silently accepting
# them as current entities.
historical_ineligible = {"BYS", "CSK", "DDR", "DHY", "HVO", "RHO", "SVU", "VDR", "YUG"}
policy_or_special = {"COK", "ESH", "GIB", "HKG", "KSV", "MSR", "NIU", "PSE", "TWN", "VAT"}
anchor_entities = {"CHN", "USA"}

universe_audit = country_master[["iso3", "country_name", "historical_entity", "first_valid_year", "last_valid_year"]].copy()
universe_audit["security_baseline_classification"] = np.select(
    [
        universe_audit["iso3"].isin(historical_ineligible),
        universe_audit["iso3"].isin(policy_or_special),
        universe_audit["iso3"].isin(anchor_entities),
        universe_audit["historical_entity"].fillna(False)
        & universe_audit["first_valid_year"].notna()
        & universe_audit["last_valid_year"].notna(),
    ],
    [
        "historical_entity_ineligible",
        "policy_or_special_entity_excluded",
        "anchor_entity",
        "historical_entity_year_conditioned",
    ],
    default="eligible_current_entity",
)
universe_audit["included_in_candidate"] = universe_audit["security_baseline_classification"].isin(
    ["eligible_current_entity", "historical_entity_year_conditioned"]
)
universe_audit["country_master_correction_proposed"] = (
    universe_audit["iso3"].isin(historical_ineligible)
    & ~universe_audit["historical_entity"].fillna(False)
)

print(universe_audit["security_baseline_classification"].value_counts().to_string())
print()
print("Country-master correction proposals:")
universe_audit.loc[
    universe_audit["country_master_correction_proposed"],
    ["iso3", "country_name", "historical_entity"],
]


security_baseline_classification
eligible_current_entity               191
policy_or_special_entity_excluded      10
historical_entity_ineligible            9
anchor_entity                           2
historical_entity_year_conditioned      1

Country-master correction proposals:


,iso3,country_name,historical_entity
28,BYS,Byelorussian SSR,False
43,CSK,Czechoslovakia,False
47,DDR,German Democratic Republic,False
49,DHY,Dahomey,False
84,HVO,Upper Volta,False
158,RHO,Rhodesia,False
178,SVU,Soviet Union,False
203,VDR,North Vietnam,False
209,YUG,Yugoslavia,False


## 3. Expand verified membership spells into annual evidence

In [4]:
START_YEAR = 2004
END_YEAR = 2025
ANCHORS = ["China", "United States"]
ANCHOR_ISO3 = {"China": "CHN", "United States": "USA"}

annual_records = []
for row in register.itertuples(index=False):
    first_year = max(START_YEAR, int(row.start_year))
    last_year = min(END_YEAR, int(row.end_year))
    for year in range(first_year, last_year + 1):
        annual_records.append({**row._asdict(), "year": year})
verified_evidence = pd.DataFrame(annual_records)

assert not verified_evidence.duplicated(["country_iso3", "anchor", "year", "arrangement_id"]).any()
assert verified_evidence["year"].between(START_YEAR, END_YEAR).all()
assert verified_evidence["country_iso3"].isin(country_master["iso3"]).all()
assert not (verified_evidence["country_iso3"] == verified_evidence["anchor"].map(ANCHOR_ISO3)).any()

verified_evidence.groupby(["anchor", "arrangement_id", "arrangement_name"], as_index=False).agg(
    country_years=("country_iso3", "size"),
    countries=("country_iso3", "nunique"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    depth=("depth", "max"),
)

,anchor,arrangement_id,arrangement_name,country_years,countries,first_year,last_year,depth
0,China,china_dprk_treaty,"China-DPRK Treaty of Friendship, Cooperation a...",22,1,2004,2025,3
1,China,china_solomon_security,China-Solomon Islands Security Cooperation Fra...,4,1,2022,2025,2
2,China,sco_full_member,Shanghai Cooperation Organisation full membership,133,9,2004,2025,1
3,United States,aukus,AUKUS,10,2,2021,2025,2
4,United States,five_eyes,Five Eyes,88,4,2004,2025,2
5,United States,mnna,Major Non-NATO Ally status and associated trea...,336,19,2004,2025,3
6,United States,nato,North Atlantic Treaty Organization,604,31,2004,2025,3
7,United States,quad,Quadrilateral Security Dialogue,33,3,2007,2025,1
8,United States,rio_treaty,Inter-American Treaty of Reciprocal Assistance,407,21,2004,2025,3
9,United States,us_india_gsomia,US-India GSOMIA,22,1,2004,2025,2


## 4. Compare the original workbook with the verified register

The comparison uses the original research workbook only to identify differences. The verified register controls the candidate score.

In [5]:
def normalise_name(value: object) -> str:
    text = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().lower()
    text = text.replace("&", " and ")
    return " ".join(re.sub(r"[^a-z0-9]+", " ", text).split())

relationship_crosswalk = pd.read_csv(RELATIONSHIP_CROSSWALK_PATH)
source_aliases = pd.read_csv(SOURCE_ALIAS_PATH)
mapping_rows = []
for row in country_master.itertuples(index=False):
    mapping_rows.append((normalise_name(row.country_name), row.iso3))
for row in relationship_crosswalk.itertuples(index=False):
    mapping_rows.append((normalise_name(row.drd_country_name), row.country))
for row in source_aliases.itertuples(index=False):
    mapping_rows.append((normalise_name(row.source_name), row.iso3))
mapping_table = pd.DataFrame(mapping_rows, columns=["normalized_name", "country_iso3"]).dropna()
assert mapping_table.groupby("normalized_name")["country_iso3"].nunique().max() == 1
mapping_lookup = mapping_table.drop_duplicates("normalized_name").set_index("normalized_name")["country_iso3"].to_dict()
manual_aliases = {
    "congo": "COG", "ivory coast": "CIV", "democratic people s republic of korea": "PRK",
    "congo democratic republic": "COD", "gambia": "GMB", "micronesia fed states of": "FSM",
    "eswatini": "SWZ", "north macedonia": "MKD",
}

workbook = load_workbook(WORKBOOK_PATH, data_only=True)
sheet = workbook["Treaties"]
workbook_countries = pd.DataFrame({
    "workbook_row": range(3, 196),
    "country_source_name": [sheet.cell(r, 1).value for r in range(3, 196)],
}).dropna(subset=["country_source_name"])
workbook_countries["normalized_name"] = workbook_countries["country_source_name"].map(normalise_name)
workbook_countries["country_iso3"] = workbook_countries["normalized_name"].map(mapping_lookup)
mask = workbook_countries["country_iso3"].isna() & workbook_countries["normalized_name"].isin(manual_aliases)
workbook_countries.loc[mask, "country_iso3"] = workbook_countries.loc[mask, "normalized_name"].map(manual_aliases)
assert workbook_countries["country_iso3"].notna().all()

workbook_to_verified = {
    "NATO": "nato",
    "Rio Treaty": "rio_treaty",
    "Major US Non-Nato & Non Rio Treaty Allies": "mnna",
    "Five Eyes": "five_eyes",
    "AUKUS": "aukus",
    "Quadrilateral Security Dialogue": "quad",
    "US India General Security of Military Information Agreement (GSOMIA)": "us_india_gsomia",
    "Treaty Allies Of China": "china_dprk_treaty",
    "Shanghai Cooperation Organisation": "sco_full_member",
}

workbook_records = []
for column in range(2, sheet.max_column + 1):
    arrangement = sheet.cell(1, column).value
    year = sheet.cell(2, column).value
    if arrangement not in workbook_to_verified or pd.isna(year):
        continue
    for country in workbook_countries.itertuples(index=False):
        workbook_records.append({
            "country_iso3": country.country_iso3,
            "anchor": "China" if arrangement in {"Treaty Allies Of China", "Shanghai Cooperation Organisation"} else "United States",
            "year": int(year),
            "arrangement_id": workbook_to_verified[arrangement],
            "workbook_active": pd.notna(sheet.cell(int(country.workbook_row), column).value),
        })
workbook_evidence = pd.DataFrame(workbook_records)

verified_active = verified_evidence[["country_iso3", "anchor", "year", "arrangement_id", "source_url"]].copy()
verified_active["verified_active"] = True
comparison_keys = ["country_iso3", "anchor", "year", "arrangement_id"]
all_keys = pd.concat([workbook_evidence[comparison_keys], verified_active[comparison_keys]]).drop_duplicates()
discrepancies = (
    all_keys
    .merge(workbook_evidence, on=comparison_keys, how="left")
    .merge(verified_active, on=comparison_keys, how="left")
)
discrepancies["workbook_active"] = discrepancies["workbook_active"].eq(True)
discrepancies["verified_active"] = discrepancies["verified_active"].eq(True)
discrepancies = discrepancies.loc[discrepancies["workbook_active"] != discrepancies["verified_active"]].copy()
discrepancies["difference_type"] = np.where(
    discrepancies["workbook_active"],
    "workbook_active_verified_inactive",
    "workbook_inactive_verified_active",
)
discrepancies["resolution"] = "verified register controls candidate"

print(f"Workbook-versus-register discrepancies: {len(discrepancies):,}")
discrepancies.groupby(["arrangement_id", "difference_type"], as_index=False).size()

Workbook-versus-register discrepancies: 189


,arrangement_id,difference_type,size
0,aukus,workbook_inactive_verified_active,4
1,china_solomon_security,workbook_inactive_verified_active,4
2,mnna,workbook_active_verified_inactive,11
3,mnna,workbook_inactive_verified_active,53
4,nato,workbook_active_verified_inactive,52
5,nato,workbook_inactive_verified_active,22
6,quad,workbook_active_verified_inactive,9
7,quad,workbook_inactive_verified_active,3
8,rio_treaty,workbook_inactive_verified_active,31


## 5. Construct the verified country-anchor-year candidate

All eligible countries are observed against the verified baseline scope. A zero means that the reviewed source register contains no qualifying relationship for that country-anchor-year; it is not a missing value.

In [6]:
eligible_countries = universe_audit.loc[
    universe_audit["included_in_candidate"],
    ["iso3", "country_name", "security_baseline_classification", "first_valid_year", "last_valid_year"],
].copy()

grid = pd.MultiIndex.from_product(
    [eligible_countries["iso3"], ANCHORS, range(START_YEAR, END_YEAR + 1)],
    names=["country_iso3", "anchor", "year"],
).to_frame(index=False)
grid = grid.merge(
    eligible_countries.rename(columns={"iso3": "country_iso3"}),
    on="country_iso3", how="left", validate="many_to_one",
)

# Current entities are eligible throughout the panel. Historical entities are
# eligible only during their recorded validity interval. This retains Serbia
# and Montenegro (SCG) for 2004-2006 and excludes it from 2007 onward.
current_mask = grid["security_baseline_classification"].eq("eligible_current_entity")
historical_mask = (
    grid["security_baseline_classification"].eq("historical_entity_year_conditioned")
    & grid["first_valid_year"].notna()
    & grid["last_valid_year"].notna()
    & grid["year"].between(grid["first_valid_year"], grid["last_valid_year"])
)
grid = grid.loc[current_mask | historical_mask].copy()
grid = grid.loc[grid["country_iso3"] != grid["anchor"].map(ANCHOR_ISO3)]
grid = grid.drop(columns=["security_baseline_classification", "first_valid_year", "last_valid_year"])

def join_unique(values: pd.Series) -> str:
    return " | ".join(sorted(set(values)))

max_depth = verified_evidence.groupby(["country_iso3", "anchor", "year"])["depth"].transform("max")
verified_evidence["controls_score"] = verified_evidence["depth"].eq(max_depth)

summary = verified_evidence.groupby(["country_iso3", "anchor", "year"], as_index=False).agg(
    maximum_depth=("depth", "max"),
    additive_depth=("depth", "sum"),
    active_arrangement_count=("arrangement_id", "nunique"),
    supporting_arrangements=("arrangement_name", join_unique),
    relationship_types=("relationship_type", join_unique),
)
controlling = (
    verified_evidence.loc[verified_evidence["controls_score"]]
    .groupby(["country_iso3", "anchor", "year"], as_index=False)
    .agg(controlling_arrangements=("arrangement_name", join_unique))
)
summary = summary.merge(controlling, on=["country_iso3", "anchor", "year"], how="left", validate="one_to_one")

panel = grid.merge(summary, on=["country_iso3", "anchor", "year"], how="left", validate="one_to_one")
for column in ["maximum_depth", "additive_depth", "active_arrangement_count"]:
    panel[column] = panel[column].fillna(0)
for column in ["supporting_arrangements", "relationship_types", "controlling_arrangements"]:
    panel[column] = panel[column].fillna("")
panel["observed"] = True
panel["status"] = np.where(panel["maximum_depth"].gt(0), "observed_active_tie", "observed_no_active_tie")
panel["security_institutional_score_raw"] = panel["maximum_depth"] / 3
panel["source_completeness_status"] = "verified_baseline_scope"
panel["method"] = "maximum_verified_security_institution_depth_divided_by_3"
panel["source"] = "verified security arrangement membership register"

candidate_columns = [
    "country_iso3", "country_name", "anchor", "year", "observed", "status",
    "maximum_depth", "active_arrangement_count", "controlling_arrangements",
    "supporting_arrangements", "relationship_types", "security_institutional_score_raw",
    "source_completeness_status", "method", "source",
]
candidate = panel[candidate_columns].copy()

candidate.groupby(["anchor", "year"], as_index=False).agg(
    eligible_rows=("country_iso3", "size"),
    active_ties=("maximum_depth", lambda x: x.gt(0).sum()),
    mean_score=("security_institutional_score_raw", "mean"),
    maximum_score=("security_institutional_score_raw", "max"),
)

,anchor,year,eligible_rows,active_ties,mean_score,maximum_score
0,China,2004,192,6,0.013889,1.0
1,China,2005,192,6,0.013889,1.0
2,China,2006,192,6,0.013889,1.0
3,China,2007,191,6,0.013962,1.0
4,China,2008,191,6,0.013962,1.0
5,China,2009,191,6,0.013962,1.0
6,China,2010,191,6,0.013962,1.0
7,China,2011,191,6,0.013962,1.0
8,China,2012,191,6,0.013962,1.0
9,China,2013,191,6,0.013962,1.0


## 6. Aggregation sensitivity and substantive asymmetry

The baseline preserves the denser US alliance system and the narrower Chinese preference for non-bloc partnerships. Asymmetry is not treated as missingness. The completeness audit asks whether qualifying formal arrangements are omitted; it does not require equal category counts.

In [7]:
panel["score_maximum_depth"] = panel["maximum_depth"] / 3
panel["score_capped_additive"] = panel["additive_depth"].clip(upper=3) / 3
panel["score_difference"] = panel["score_capped_additive"] - panel["score_maximum_depth"]

method_sensitivity = panel.groupby(["anchor", "year"], as_index=False).agg(
    eligible_rows=("country_iso3", "size"),
    active_rows=("maximum_depth", lambda x: x.gt(0).sum()),
    mean_maximum_depth_score=("score_maximum_depth", "mean"),
    mean_capped_additive_score=("score_capped_additive", "mean"),
    changed_by_additive_rule=("score_difference", lambda x: x.gt(0).sum()),
    maximum_score_difference=("score_difference", "max"),
)

source_completeness = register.groupby(
    ["anchor", "arrangement_id", "arrangement_name", "review_status"], as_index=False
).agg(
    membership_spells=("country_iso3", "size"),
    countries=("country_iso3", "nunique"),
    first_year=("start_year", "min"),
    last_year=("end_year", "max"),
    source_count=("source_url", "nunique"),
)

print("Verified active country-years by anchor:")
print(candidate.loc[candidate["status"].eq("observed_active_tie")].groupby("anchor").size().to_string())
method_sensitivity

Verified active country-years by anchor:
anchor
China             159
United States    1336


,anchor,year,eligible_rows,active_rows,mean_maximum_depth_score,mean_capped_additive_score,changed_by_additive_rule,maximum_score_difference
0,China,2004,192,6,0.013889,0.013889,0,0.000000
1,China,2005,192,6,0.013889,0.013889,0,0.000000
2,China,2006,192,6,0.013889,0.013889,0,0.000000
3,China,2007,191,6,0.013962,0.013962,0,0.000000
4,China,2008,191,6,0.013962,0.013962,0,0.000000
5,China,2009,191,6,0.013962,0.013962,0,0.000000
6,China,2010,191,6,0.013962,0.013962,0,0.000000
7,China,2011,191,6,0.013962,0.013962,0,0.000000
8,China,2012,191,6,0.013962,0.013962,0,0.000000
9,China,2013,191,6,0.013962,0.013962,0,0.000000


## 7. Export revised development outputs

In [8]:
exports = {
    "security_country_universe_audit.csv": universe_audit,
    "security_membership_discrepancy_audit.csv": discrepancies,
    "security_institutional_evidence_verified_long.csv": verified_evidence,
    "security_institutional_method_sensitivity_verified.csv": method_sensitivity,
    "security_institutional_source_completeness.csv": source_completeness,
    "security_institutional_score_candidate_verified.csv": candidate,
}
for filename, frame in exports.items():
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    print(f"Wrote {path.relative_to(PROJECT_ROOT)} ({len(frame):,} rows)")

Wrote outputs/tables/security_institutional_ties/security_country_universe_audit.csv (213 rows)
Wrote outputs/tables/security_institutional_ties/security_membership_discrepancy_audit.csv (189 rows)
Wrote outputs/tables/security_institutional_ties/security_institutional_evidence_verified_long.csv (1,659 rows)
Wrote outputs/tables/security_institutional_ties/security_institutional_method_sensitivity_verified.csv (44 rows)
Wrote outputs/tables/security_institutional_ties/security_institutional_source_completeness.csv (10 rows)
Wrote outputs/tables/security_institutional_ties/security_institutional_score_candidate_verified.csv (8,410 rows)


## 8. Final validation and provisional recommendation

**Provisional recommendation:** retain maximum verified depth divided by three as the Security Institutional candidate. Preserve supporting arrangements and relationship types. Keep SIPRI as a separate operational-security component until the final Security pillar is designed.

### Remaining matters before production sign-off

- Decide whether the Rio Treaty should remain depth `3` or move to depth `2` in the baseline taxonomy.
- Adopt an index-wide policy for Taiwan, Palestine, Kosovo, Hong Kong, Holy See, and other special entities.
- Apply the proposed historical-entity corrections to the canonical country master rather than maintaining component-specific exceptions indefinitely.
- Revisit the China-side completeness audit when new formal bilateral security agreements are identified. Generic strategic partnerships remain excluded.
- Review the full reproducible SIPRI implementation before security-pillar aggregation.

In [9]:
keys = ["country_iso3", "anchor", "year"]
assert not candidate.duplicated(keys).any()
assert candidate["year"].between(START_YEAR, END_YEAR).all()
assert candidate["security_institutional_score_raw"].between(0, 1).all()
assert set(candidate["security_institutional_score_raw"].unique()).issubset({0.0, 1/3, 2/3, 1.0})
assert candidate.loc[candidate["status"].eq("observed_no_active_tie"), "security_institutional_score_raw"].eq(0).all()
assert candidate.groupby(["anchor", "year"])["security_institutional_score_raw"].max().eq(1).all()
assert not ((verified_evidence["arrangement_id"].eq("nato")) & (verified_evidence["country_iso3"].eq("HND"))).any()
assert verified_evidence.loc[(verified_evidence["arrangement_id"].eq("nato")) & (verified_evidence["country_iso3"].eq("ALB")), "year"].min() == 2009
assert verified_evidence.loc[(verified_evidence["arrangement_id"].eq("nato")) & (verified_evidence["country_iso3"].eq("SWE")), "year"].min() == 2024
assert verified_evidence.loc[(verified_evidence["arrangement_id"].eq("aukus")), "year"].min() == 2021
assert verified_evidence.loc[(verified_evidence["arrangement_id"].eq("sco_full_member")), "depth"].eq(1).all()
assert not register["arrangement_id"].str.contains("observer|dialogue", case=False, regex=True).any()
assert all((OUTPUT_DIR / filename).exists() for filename in exports)
assert set(candidate.loc[candidate["country_iso3"].eq("SCG"), "year"]) == {2004, 2005, 2006}
assert len(candidate) == 8410
assert candidate.groupby("anchor").size().eq(4205).all()

print("Revised Notebook 12 checks passed.")
print("Candidate method: maximum verified security-institution depth divided by 3.")
print("No final combined Security pillar score has been selected or exported.")

Revised Notebook 12 checks passed.
Candidate method: maximum verified security-institution depth divided by 3.
No final combined Security pillar score has been selected or exported.
